# 05 — Final test session (§0.7 — run ONCE)

**The test set is evaluated once, at the end, with the val-selected
checkpoint of each stream.** Do not open this notebook until EVERY
stream has its selection artifact — the readiness cell below asserts
it. Every cell accesses the test set through the logging harness, so
each execution is appended to `test_invocations.jsonl`: re-runs are
visible forever. If a cell crashes, fixing and re-running is fine (the
log is the audit trail); peeking-and-tuning is not.

Streams (§6): C0 (SHARP decision fusion, P1) · C1/C2 end-to-end ·
C1-lin/C2-lin · C3/C4 (phase-B selected checkpoints) — all on the
frozen splits, all through the same harness (§0.4).


## Environment setup

Same conventions as notebook `03`: locate or clone the repo, install
requirements, mount Drive, stage the data zips into `/content/data`
using `configs/paths.yaml`. Checkpoints are read from `ckpt_root` on
Drive; every artifact produced here is written next to them.


In [ ]:
from pathlib import Path
import subprocess
import sys
import torch
import zipfile

REPO_URL = "https://github.com/FilippoIsoni/sharp-har.git"

# Locate or clone the repository root containing the sharp_har package.
# In Colab, the notebook may run from a temporary working directory.
cwd = Path.cwd().resolve()
if (cwd / "sharp_har").exists():
    REPO_DIR = cwd
elif (cwd.parent / "sharp_har").exists():
    REPO_DIR = cwd.parent
elif (cwd.parent.parent / "sharp_har").exists():
    REPO_DIR = cwd.parent.parent
else:
    REPO_DIR = Path("/content/sharp-har")
    if not REPO_DIR.exists():
        REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

# After the clone, so the file actually exists on a fresh runtime.
!pip install -q -r {REPO_DIR}/requirements.txt

sys.path.insert(0, str(REPO_DIR))

from google.colab import drive
from sharp_har.utils import read_yaml

paths_cfg = read_yaml(REPO_DIR / "configs" / "paths.yaml")
drive_root = Path(paths_cfg["drive_root"])
stage_dir = Path(paths_cfg["stage_dir"])
CKPT_ROOT = Path(paths_cfg["ckpt_root"])

# Mount Drive unconditionally (idempotent): checkpoints live on Drive
# even when the data is already staged on the VM.
drive.mount("/content/drive")

# Stage the zip archives if needed (same convention as 00_setup_smoke).
if not (stage_dir.exists() and any(stage_dir.rglob("*.txt"))):
    stage_dir.mkdir(parents=True, exist_ok=True)
    for zip_name in paths_cfg["zips"]:
        src = drive_root / zip_name
        dst = Path("/content") / zip_name
        print(f"copying {src} -> {dst}")
        subprocess.run(["cp", str(src), str(dst)], check=True)
        with zipfile.ZipFile(dst) as zf:
            zf.extractall(stage_dir)

print("Repo dir:", REPO_DIR)
print("GPU available:", torch.cuda.is_available())
print("Stage dir:", stage_dir)
print("Checkpoint root:", CKPT_ROOT)


## Readiness check

Asserts that every stream has its val-selected artifact **before any
test access happens**. Nothing below runs if something is missing.


In [ ]:
required = {
    "C0 best.ckpt": CKPT_ROOT / "C0" / "best.ckpt",
    "C1 best.ckpt": CKPT_ROOT / "C1" / "best.ckpt",
    "C2 best.ckpt": CKPT_ROOT / "C2" / "best.ckpt",
    "C1-lin head": CKPT_ROOT / "C1" / "probe_head_best.npz",
    "C2-lin head": CKPT_ROOT / "C2" / "probe_head_best.npz",
    "C3 phase-B selection": CKPT_ROOT / "C3" / "phase_b_selection.json",
    "C4 phase-B selection": CKPT_ROOT / "C4" / "phase_b_selection.json",
}
missing = [name for name, p in required.items() if not p.exists()]
assert not missing, (
    f"missing: {missing} — §0.7: the final session runs once, after ALL "
    "streams have a val-selected checkpoint (notebook 03 for training, "
    "04 for probes/phase B)."
)
print("all streams ready")


## C0 — SHARP-repo evaluation (P1)

`evaluate_c0` fixes the paper's decision fusion (TMC §4.2) and logs the
test access like every other stream (§0.7).


In [ ]:
from sharp_har.harness import evaluate_c0

c0_metrics = evaluate_c0(
    CKPT_ROOT / "C0" / "best.ckpt",
    REPO_DIR / "splits" / "p1_sharp.json",
    "test",
    stage_dir=stage_dir,
    repo_dir=REPO_DIR,
)
c0_metrics


## C1, C2 — end-to-end (P2, softmax averaging)


In [ ]:
from sharp_har.harness import evaluate

P2_SPLIT = REPO_DIR / "splits" / "p2_lab.json"

e2e_metrics = {}
for run in ("C1", "C2"):
    e2e_metrics[run] = evaluate(
        CKPT_ROOT / run / "best.ckpt", P2_SPLIT, "test",
        stage_dir=stage_dir, repo_dir=REPO_DIR,
    )
e2e_metrics


## C1-lin, C2-lin, C3, C4 — linear-probe streams (P2)

Test features are extracted now (a logged test access, §0.7) and the
heads persisted by notebook `04` are applied through
`evaluate_features` — same fusion, metrics and logging as every other
stream.


In [ ]:
import json
import numpy as np
from sharp_har.harness import cache_features, evaluate_features

streams = {
    "C1_lin": (CKPT_ROOT / "C1" / "best.ckpt", CKPT_ROOT / "C1" / "probe_head_best.npz"),
    "C2_lin": (CKPT_ROOT / "C2" / "best.ckpt", CKPT_ROOT / "C2" / "probe_head_best.npz"),
}
for run in ("C3", "C4"):
    sel = json.loads((CKPT_ROOT / run / "phase_b_selection.json").read_text())
    streams[run] = (Path(sel["selected_checkpoint"]), Path(sel["selected_head"]))

probe_metrics = {}
for run_name, (ckpt, head_path) in streams.items():
    npz = cache_features(ckpt, P2_SPLIT, "test", stage_dir=stage_dir, repo_dir=REPO_DIR)
    probe_metrics[run_name] = evaluate_features(
        npz, np.load(head_path), "test", run_name=run_name, repo_dir=REPO_DIR,
    )
probe_metrics


## Comparison table and confusion matrices

Per-AR-set metrics for every stream (§9: never aggregate-only).
Remember §0.5: differences under ~2 points are "comparable", not
improvements. C0 runs on P1, so it covers different AR-sets (NaN in
the others).


In [ ]:
from sharp_har.viz import metrics_table, plot_confusion

all_metrics = {"C0": c0_metrics, **e2e_metrics, **probe_metrics}
metrics_table(all_metrics)


In [ ]:
metrics_table(all_metrics, metric="accuracy")


In [ ]:
for run_name, metrics_csv in all_metrics.items():
    confusion_csv = Path(metrics_csv).with_name(
        Path(metrics_csv).name.replace("_metrics.csv", "_confusion.csv"))
    plot_confusion(confusion_csv);


## Commit the measured artifacts

Copies every per-AR-set test CSV and the test-invocation logs into the
repo (`reports/final/`, prefixed with the run folder name) so they can
be committed. Then, in the SAME commit: `reports/final/`, this executed
notebook under `notebooks/runs/` (verbatim, with outputs), and the
`STATUS.md` update.


In [ ]:
import shutil

final_dir = REPO_DIR / "reports" / "final"
final_dir.mkdir(parents=True, exist_ok=True)

copied = []
for run_folder in ("C0", "C1", "C2", "C3", "C4"):
    d = CKPT_ROOT / run_folder
    for f in sorted(list(d.glob("*_test_*.csv")) + list(d.glob("test_invocations.jsonl"))):
        dst = final_dir / f"{run_folder}_{f.name}"
        shutil.copy2(f, dst)
        copied.append(dst.name)
print(f"{len(copied)} files -> {final_dir}")
copied
